In [1]:
from openai import OpenAI
from dotenv import load_dotenv
from glob import glob
import os
import json
import base64

load_dotenv()
strAPI_key = os.getenv("OPENAI_API_KEY")    # Load OpenAI API Key
client = OpenAI(api_key=strAPI_key)         # Create OpenAI Client Instance 

def encode_image(image_path):               # Encode image to base64 string
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")

In [2]:
def image_quiz(image_path, n_trial=0, max_trial=3):
    if n_trial >= max_trial:    # If Max trial exceeded Then raise Exception
        return None, False      # Return tuple instead of raising Exception
        
    try:    
        base64_image = encode_image(image_path) # Encode image to base64 string

        quiz_prompt = """
        제공된 이미지를 바탕으로, 다음과 같은 형식의 퀴즈를 만들어주세요. 
        정답은 1~4 중 하나만 해당하도록 출제하세요.
        토익 리스닝 문제 스타일로 문제를 만들어주세요.
        아래는 예시입니다. 
        ----- 예시 -----

        [Quiz]: 다음 이미지에 대한 설명 중 옳지 않은 것은 무엇인가요?
        - (1) 베이커리에서 사람들이 빵을 사고 있는 모습이 담겨 있습니다.
        - (2) 맨 앞에 서 있는 사람은 빨간색 셔츠를 입고 있습니다.
        - (3) 기차를 타기 위해 줄을 서 있는 사람들이 있습니다.
        - (4) 점원은 노란색 티셔츠를 입고 있습니다.
            
        Listening: Which of the following descriptions of the image is incorrect?
        - (1) It shows people buying bread at a bakery.
        - (2) The person standing at the front is wearing a red shirt.
        - (3) There are people lining up to take a train.
        - (4) The clerk is wearing a yellow T-shirt.

        [Answer]: (4) 점원은 노란색 티셔츠가 아닌 파란색 티셔츠를 입고 있습니다.
        ======
        """
        strMessages = [
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": quiz_prompt},
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": f"data:image/jpeg;base64,{base64_image}",
                            },
                        },
                    ],
                }
            ]

        try:
            response = client.chat.completions.create(
                    model="gpt-4o",         # Specify GPT-4o model
                    messages = strMessages  # Sent strMessages to messages parameter
            )
        except Exception as e:
            print(f"Fail image creation: {e}....\n" )
            return image_quiz(image_path, n_trial + 1)

        content = response.choices[0].message.content

        if "Listening:" in content:
            return content, True
        else:
            return image_quiz(image_path, n_trial+1)
    except Exception as e:
        print(f"Fail image quiz creation: {e}")
        return image_quiz(image_path, n_trial + 1, max_trial)    

In [ ]:
MD_Text = ''                            # Initialize markdown text
P_No = 1                                # Problem number start
for g in glob('../Data/*.jpg'):   # Get all jpg files in Path
    Quiz, is_suceed = image_quiz(g)

    if not is_suceed:
        print(f"Skipping {g}, quiz generation failed.")
        continue

    divider = f'\n ## Quiz-{P_No} \n'   # Fotmat problem number
    print(divider)
    
    MD_Text += divider                  # Append problem number to markdown text
    filename = os.path.basename(g)      # Set image file path
    MD_Text += f'![image]({filename})\n\n'  # Append image link to markdown text

    print(Quiz)                            # Print quiz  
    MD_Text += Quiz + '\n\n---------------------\n\n'
    
    # Write markdown text to file
    with open('../Data/Image_quiz.md', 'w', encoding='utf-8') as f:
        f.write(MD_Text)
    
    P_No += 1                           # Increment problem number


 ## Quiz-1 

[Quiz]: 다음 이미지에 대한 설명 중 옳지 않은 것은 무엇인가요?
- (1) 보트가 물 위에 떠 있는 모습입니다.
- (2) 하늘을 나는 새들이 많이 보입니다.
- (3) 사진은 맑은 날씨에 찍혔습니다.
- (4) 사진은 흑백 스타일로 촬영되었습니다.

Listening: Which of the following descriptions of the image is incorrect?
- (1) A boat is floating on the water.
- (2) Many birds are flying in the sky.
- (3) The photo was taken on a clear day.
- (4) The photo is captured in black and white.

[Answer]: (3) 사진은 맑은 날씨가 아닌 안개가 낀 날에 찍혔습니다.

 ## Quiz-2 

[Quiz]: 다음 이미지에 대한 설명 중 옳지 않은 것은 무엇인가요?
- (1) 요트가 선착장에 정박해 있는 모습입니다.
- (2) 배경에는 큰 건물이 보입니다.
- (3) 호수 한가운데에 노란색 택시 보트가 있습니다.
- (4) 사람들이 해변에서 수영하고 있습니다.

Listening: Which of the following descriptions of the image is incorrect?
- (1) There are yachts docked at the pier.
- (2) There is a large building in the background.
- (3) A yellow taxi boat is in the middle of the water.
- (4) People are swimming at the beach.

[Answer]: (4) 사람들이 해변에서 수영하고 있지 않습니다.
